# 02 - Baseline models

A Response LightGBM classifier: predicts `P(conversion | X)` directly,
ignoring treatment. This is **not** a causal estimator -- it answers "who is
likely to convert," not "who converts *because of* treatment." It's the
non-causal targeting comparator the uplift models in `03_uplift_models.ipynb`
and `04_causal_forest.ipynb` are compared against.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

from src.data import PRIMARY_OUTCOME, TREATMENT_COLUMN, load_parquet
from src.preprocessing import LightGBMFeatureTransform
from src.models import fit_response_model, predict
from src.evaluation import evaluate_ranking, response_diagnostics

PROCESSED_DIR = REPO_ROOT / "data" / "processed"
train_frame = load_parquet(PROCESSED_DIR / "train.parquet")
val_frame = load_parquet(PROCESSED_DIR / "validation.parquet")
train_frame.shape, val_frame.shape

## Feature representation

Continuous features stay `float64`; categorical features become a
train-fitted pandas categorical dtype so LightGBM uses native categorical
splits instead of treating the token as an ordered number. Fit on TRAIN only,
then reused unchanged on validation.

In [ ]:
transform = LightGBMFeatureTransform()
X_train = transform.fit_transform(train_frame)
X_val = transform.transform(val_frame)

Y_train, Y_val = train_frame[PRIMARY_OUTCOME], val_frame[PRIMARY_OUTCOME]
T_train, T_val = train_frame[TREATMENT_COLUMN], val_frame[TREATMENT_COLUMN]

## Fit

In [ ]:
response_model = fit_response_model(X_train, Y_train, X_val, Y_val, seed=42)
response_model.best_iteration

## Evaluate

Two different questions, on purpose: response-model diagnostics (does it
predict conversion well?) versus ranking as an uplift proxy (does ranking by
predicted response probability also rank well by *incremental* conversion?
Response probability is not a causal score, so this is expected to
underperform genuine uplift models).

In [ ]:
val_scores = predict(response_model, X_val)
diagnostics = response_diagnostics(val_scores, Y_val)
diagnostics

In [ ]:
ranking = evaluate_ranking(val_scores, T_val, Y_val)
print("qini_above_random:", round(ranking.qini_above_random, 6))
ranking.uplift_at_k

In [ ]:
import matplotlib.pyplot as plt

plt.plot(ranking.qini_curve["coverage"], ranking.qini_curve["qini_gain"], label="Response model")
plt.plot([0, 1], [0, ranking.theoretical_random_qini_area * 2], "--", label="Theoretical random")
plt.xlabel("Population coverage")
plt.ylabel("Cumulative incremental conversions")
plt.legend()
plt.title("Response model: Qini curve")

## Next

Continue with `03_uplift_models.ipynb` for T-Learner and X-Learner.